# RAG Evaluation: Measuring What Matters

## You've Built the Pipeline... But Does It Work?

In **Tutorial 1A**, you learned how to chunk documents.  
In **Tutorial 1B**, you learned how to optimize queries.  
In **Tutorial 2A** and **2B**, you learned retrieval methods and smart orchestration.  
In **Tutorial 3**, you learned post-retrieval context preparation.

Now comes the critical question: **How do you know any of it actually helped?**

---

## The Journey So Far

```
Tutorial 1A: Chunking ✅
    ↓
Tutorial 1B: Query Optimization ✅
    ↓
Tutorial 2A: Core Retrieval Methods ✅
    ↓
Tutorial 2B: Smart Orchestration ✅
    ↓
Tutorial 3: Post-Retrieval ✅
    ↓
Tutorial 4: Evaluation ← You are here
    ↓
Next: Agentic RAG
```

**Your pipeline runs. Evaluation tells you where it breaks.**

---

## The Challenge

A RAG system can fail at **any stage** - and a single accuracy number hides all of it:

- ✅ Retrieval returns chunks... but are they the *right* chunks?
- ✅ Generation sounds fluent... but is it *faithful* to the evidence?
- ✅ The agent picks a tool... but was it the *right* one?

**The question:** Which layer failed - retrieval, generation, tools, or the full stack?

---

## What You'll Learn

We'll walk through a **layered evaluation stack** that mirrors the pipeline you built:

**The Layers:**
1. **Retrieval Evaluation** - Hit@k, MRR, keyword overlap, RAGAS context metrics
2. **Generation Evaluation** - Faithfulness, answer accuracy
3. **Tool / Agent Evaluation** - Tool selection and routing accuracy
4. **End-to-End Pipeline** - One-call scorecard across all stages

**Two metric families:**
| Family | Best for |
|--------|----------|
| **Classical / Custom** | Fast, deterministic baselines (Hit@k, MRR, keyword overlap) |
| **RAGAS** | LLM-as-judge semantic metrics (faithfulness, context precision, …) |

---

## Tutorial Structure

```
Part 1: Load Test Data & Run Baseline RAG
    ↓
Part 2: Layer 1 - Retrieval Evaluation
    ↓
Part 3: Layer 2 - Generation Evaluation
    ↓
Part 4: Layer 3 - Tool / Agent Evaluation
    ↓
Part 5: End-to-End Pipeline
    ↓
Part 6: Baseline vs Pre / Mid / Post (independent sections)
    ↓
Part 7: Hands-On - Evaluate Your Own Query
```

**Our approach:** Score each layer independently, then read the stack as a diagnostic - not one number.

Code lives in **`retrieval_playground/src/evaluation/`**.

Let's measure what matters! 🚀


## ⚙️ Setup: Imports

We wire up the same building blocks from earlier tutorials:

| Component | Source | Role in this notebook |
|-----------|--------|----------------------|
| `RAG` | `baseline_rag.py` | Runs retrieve → generate for each test query |
| `ChunkingStrategy` | pre-retrieval | Same chunking config as tutorial 1 |
| `semantic_layer` | pre-retrieval routing (Tutorial 1B) | Demo for tool-selection evaluation in a RAG router |
| `ToolEvaluator` / `ToolTrace` | `tool_metrics.py` | Score agent-style tool decisions from logged traces |
| Evaluators | `src/evaluation/` | Score each pipeline stage |

> **Note:** Logging is suppressed so live demo output stays readable during presentation.


In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

import retrieval_playground.utils.ragas_compat

import json
from typing import Dict, List, Any

import pandas as pd

from retrieval_playground.utils import config
from retrieval_playground.src.baseline_rag import RAG
from retrieval_playground.src.pre_retrieval.routing import semantic_layer
from retrieval_playground.src.evaluation import (
    RetrievalEvaluator,
    GenerationEvaluator,
    ToolEvaluator,
    ToolTrace,
    RAGEvaluationPipeline,
)

## 📂 Load Test Data & Run Baseline RAG

Every evaluation metric needs a **ground truth** to compare against. Our test set (`evaluation_dataset.json`) provides two gold labels per query:

| Field | What it is | Example use |
|-------|------------|---------------|
| `user_input` | The question | Drives retrieval & generation |
| `reference_context` | Gold **evidence** passage | Classical retrieval metrics |
| `reference` | Gold **answer** | Generation metrics & RAGAS context metrics |

> ⚠️ **Important distinction:** Classical retrieval metrics compare chunks → `reference_context`. RAGAS context_precision / context_recall compare chunks → `reference` (the answer). Mixing these up gives misleading scores.

We run baseline RAG on a **small slice** (3 queries) so live demos finish quickly - swap in the full test set for production benchmarking.


In [ ]:
def load_test_queries() -> List[Dict[str, Any]]:
    queries_path = config.TEST_DATA_DIR / "evaluation_dataset.json"
    with open(queries_path, "r") as f:
        return json.load(f)

strategy = "recursive_character"
rag = RAG(strategy=strategy)

# Use a small slice for the tutorial demo
test_queries = load_test_queries()[:3]
questions = [q["user_input"] for q in test_queries]
ground_truths = [q["reference"] for q in test_queries]
reference_contexts = [q["reference_context"] for q in test_queries]

rag_results = []
retrieved_contexts = []

for query in test_queries:
    result = rag.query(query["user_input"])
    rag_results.append(result)
    retrieved_contexts.append([ctx["content"] for ctx in result["context"]])

answers = [r["answer"] for r in rag_results]

print(f"Evaluating {len(test_queries)} queries")

---

# Layer 1: Retrieval Evaluation

## The Question

> *"Did the retriever surface evidence that actually supports the answer?"*

If retrieval fails, **no amount of prompt engineering fixes generation**. This layer catches problems before they reach the LLM.

---

## Metric Definitions

Dataset Example:

- **Query:** `What is a TPU used for?`
- **Gold evidence (`reference_context`):** `TPUs accelerate dense matrix ops for neural-network training.`
- **Retrieved top-3:**
  1. `GPUs are common for deep learning workloads.`
  2. `TPUs accelerate dense matrix ops for neural-network training.`  ← relevant
  3. `Pandas is popular for tabular data prep.`

### Hit Rate @ k (classical)

**Definition:** `1` if **any** of the top-*k* chunks is relevant to the gold evidence, else `0`. Averaged over queries.

**Example (k=3):** chunk #2 matches gold → **Hit@3 = 1.0**.  
If only chunk #1 and #3 were retrieved → **Hit@3 = 0.0**.

### MRR - Mean Reciprocal Rank (classical)

**Definition:** If the first relevant chunk is at rank *r*, score is `1/r`. No hit → `0`. Averaged over queries.

**Example:** first relevant chunk is rank 2 → **MRR = 1/2 = 0.5**.  
If it were rank 1 → **MRR = 1.0**.

### Keyword Overlap (custom / lexical)

**Definition:** Fraction of gold-evidence tokens that also appear in the retrieved chunks (lexical overlap).

**Example:** gold tokens ≈ `{tpus, accelerate, dense, matrix, ops, neural, network, training}`  
Chunk #2 covers most of them → **high overlap (~0.8–1.0)**.  
Only chunk #1 ("GPUs…") → **low overlap**.

> Classical metrics compare retrieved chunks ↔ **`reference_context`** (gold evidence passage).

### Context Precision (RAGAS)

**Definition:** Of the chunks you retrieved, what fraction are useful for producing the **gold answer**? (LLM-as-judge; noise penalty.)

**Example**

- **Query:** `What is a TPU used for?`
- **Gold answer (`reference`):** `TPUs speed up neural-network training via matrix operations.`
- **Contexts:**
  - C1: `TPUs accelerate matrix ops for neural-network training.` ← useful
  - C2: `The weather in Austin was rainy last week.` ← noise

**Score intuition:** 1 useful / 2 retrieved → **context precision ≈ 0.5**.

### Context Recall (RAGAS)

**Definition:** Of the claims / facts needed for the gold answer, what fraction are covered by the retrieved contexts?

**Example**

- **Gold answer claims:** (1) TPUs speed up training, (2) via matrix operations.
- **Only retrieved:** `TPUs are Google accelerators.`  
  → covers hardware name, misses both claims → **low context recall**.
- **Retrieved:** `TPUs speed up NN training using matrix ops.`  
  → both claims present → **high context recall (~1.0)**.

> RAGAS context metrics compare retrieved chunks ↔ **`reference`** (gold **answer**), not the evidence passage.

| Metric | Type | Range | What it tells you | Low score means… |
|--------|------|-------|-------------------|------------------|
| **Hit Rate @ k** | Classical | 0–1 | Any relevant chunk in top-k? | Retriever never surfaces gold evidence |
| **MRR** | Classical | 0–1 | How early does the first relevant chunk appear? | Right chunk exists but ranks too low |
| **Keyword Overlap** | Custom (lexical) | 0–1 | Token overlap with gold evidence | Semantic match without lexical overlap |
| **Context Precision** | RAGAS | 0–1 | Fraction of retrieved chunks that matter | Too much noise in the context window |
| **Context Recall** | RAGAS | 0–1 | Fraction of needed evidence retrieved | Gold facts missing from retrieval |

See **`retrieval_metrics.py`** and **`ragas_runner.py`** for implementations.

After scoring the default collection, compare all **4 chunking strategies** from Tutorial 1A on the same queries.


In [ ]:
retrieval_evaluator = RetrievalEvaluator(k=3)

# Classical metrics — compare chunks to gold evidence (reference_context)
classical = retrieval_evaluator.evaluate(
    questions=questions,
    contexts=retrieved_contexts,
    reference_contexts=reference_contexts,
)
print("=== Classical Retrieval Metrics ===")
for name, score in classical.scores.items():
    print(f"  {name}: {score:.3f}")

# RAGAS retrieval metrics — compare chunks to gold ANSWERS (reference)
ragas_retrieval = retrieval_evaluator.evaluate_ragas(
    questions=questions,
    contexts=retrieved_contexts,
    reference_answers=ground_truths,
    answers=answers,
)
print("\n=== RAGAS Retrieval Metrics (vs reference answers) ===")
for name, score in ragas_retrieval.items():
    print(f"  {name}: {score:.3f}")

### Compare All 4 Chunking Strategies

Chunking changes what lands in the index. Run the same queries against every collection from Tutorial 1A and compare:

| Family | Metrics | Compared to |
|--------|---------|-------------|
| **Classical** | Hit@k, MRR, keyword overlap | `reference_context` (gold evidence) |
| **RAGAS** | Context precision, context recall | `reference` (gold answer) — needs a generated answer per strategy |

| Strategy | Collection | Best when… |
|----------|------------|------------|
| Recursive Character | `recursive_character` | Fast baseline, general text |
| Contextual | `contextual` | Chunks need document-level context |
| Parent-Child | `parent_child` | Small precise child hits, large parent context |
| Docling | `docling` | PDFs with tables, images, structure |

> ⏱️ Generates an answer per strategy × query, then runs RAGAS — slower than classical-only.


In [ ]:
from retrieval_playground.src.post_retrieval import document_assembly

CHUNKING_STRATEGIES = {
    "recursive_character": "Recursive Character",
    "contextual": "Contextual",
    "parent_child": "Parent-Child",
    "docling": "Docling",
}

chunking_evaluator = RetrievalEvaluator(k=3)
chunking_rows = []

for collection, label in CHUNKING_STRATEGIES.items():
    strategy_contexts = []
    strategy_questions = []
    strategy_refs = []
    strategy_answers = []
    strategy_gold_answers = []

    print(f"\n--- {label} ({collection}) ---")
    for query_data in test_queries:
        query = query_data["user_input"]
        docs = [
            doc
            for doc, _ in rag.retrieve_context(query, k=3, collection_name=collection)
        ]
        contexts = [doc.page_content for doc in docs]
        answer = document_assembly.generate_answer(query, docs)

        strategy_contexts.append(contexts)
        strategy_questions.append(query)
        strategy_refs.append(query_data["reference_context"])
        strategy_answers.append(answer)
        strategy_gold_answers.append(query_data["reference"])

    classical = chunking_evaluator.evaluate(
        strategy_questions, strategy_contexts, strategy_refs
    )
    ragas_scores = chunking_evaluator.evaluate_ragas(
        questions=strategy_questions,
        contexts=strategy_contexts,
        reference_answers=strategy_gold_answers,
        answers=strategy_answers,
    )

    chunking_rows.append({
        "Chunking Strategy": label,
        **{k: round(v, 3) for k, v in classical.scores.items()},
        **{k: round(v, 3) for k, v in ragas_scores.items()},
    })

chunking_df = pd.DataFrame(chunking_rows)
print("\n=== Retrieval by Chunking Strategy (classical + RAGAS) ===")
chunking_df


---

# Layer 2: Generation Evaluation

## The Question

> *"Given the retrieved context, is the answer actually good?"*

Retrieval can be perfect and generation can still fail — by hallucinating facts, ignoring the question, or being uselessly verbose.

---

## Metric Definitions

Dataset Example:

- **Query:** `What is a TPU used for?`
- **Retrieved context:** `TPUs accelerate dense matrix ops for neural-network training.`
- **Gold answer (`reference`):** `TPUs speed up neural-network training via matrix operations.`

### Faithfulness (RAGAS)

**Definition:** Are the claims in the **generated answer** supported by the **retrieved context**?  
Does **not** use ground truth - only answer ↔ context.

**Example A (faithful):**  
Generated: `TPUs accelerate matrix ops for neural-network training.`  
→ claims ⊆ context → **high faithfulness (~1.0)**.

**Example B (unfaithful):**  
Generated: `TPUs were invented in 2010 and replace all CPUs.`  
→ invented claims not in context → **low faithfulness**.

### Answer Accuracy (RAGAS / NVIDIA)

**Definition:** Does the generated answer match the **gold reference** (semantic / factual agreement)?

**Example A:**  
Generated ≈ gold ("speed up training via matrix ops") → **high accuracy**.

**Example B:**  
Generated: `TPUs are mainly for web search ranking.`  
→ fluent but wrong vs gold → **low accuracy**.

> Faithfulness can be high while accuracy is low: the model stayed grounded in *wrong / incomplete* context.

### Answer Length Ratio (custom)

**Definition:** `len(generated) / len(gold)` (character- or token-based length ratio).  
Near **1.0** ≈ similar verbosity; **≫ 1** = much longer than gold; **≪ 1** = much shorter.

**Example:** gold = 80 chars, generated = 400 chars → **ratio ≈ 5.0** (verbose).

| Metric | Type | What it measures | Low / extreme score means… |
|--------|------|------------------|----------------------------|
| **Faithfulness** | RAGAS | Claims supported by context? | Hallucination vs retrieved docs |
| **Answer Accuracy** | RAGAS (NVIDIA) | Match to gold reference? | Wrong or incomplete vs gold |
| **Answer Length Ratio** | Custom | Conciseness vs gold | ≫1 too verbose; ≪1 too terse |

See **`generation_metrics.py`** for implementations.


In [ ]:
generation_evaluator = GenerationEvaluator(
    ragas_metrics=["faithfulness", "answer_accuracy"]
)
generation_result = generation_evaluator.evaluate(
    questions=questions,
    answers=answers,
    contexts=retrieved_contexts,
    ground_truths=ground_truths,
)

print("=== RAGAS Generation Metrics ===")
for name in ["faithfulness", "answer_accuracy"]:
    if name in generation_result.ragas_scores:
        print(f"  {name}: {generation_result.ragas_scores[name]:.3f}")

print("\n=== Custom Generation Metrics ===")
for name, score in generation_result.custom_scores.items():
    print(f"  {name}: {score:.3f}")

## The Failure Modes

| Symptom | Likely culprit | Metric to check | Where to fix |
|---------|---------------|-----------------|--------------|
| Answer cites facts not in docs | Hallucination | ↓ Faithfulness | Post-retrieval grading, stricter prompts |
| Answer sounds good but disagrees with gold | Factual mismatch | ↓ Answer Accuracy | Retrieval coverage, model choice |
| Answer is a wall of text | Verbosity | ↑ Length Ratio | Context compression (Tutorial 3) |

> 💡 **Presentation tip:** Faithfulness and answer accuracy are *independent*. A model can be faithful (no hallucination) or accurate-looking yet unfaithful to the retrieved context.

---

# Layer 3: Tool / Agent Evaluation

## The Question

> *"Did the system pick the right tool, execute it successfully, and use the result appropriately?"*

Modern RAG is rarely a single retrieve → generate path. **Agents** break a user request into steps, choosing among tools at each step — vector search, SQL, web search, calculators, and more. A wrong tool choice upstream poisons everything downstream, even when retrieval and generation metrics look fine.

---

## Metric Definitions (with mini examples)

### Tool Selection Accuracy

**Definition:** Fraction of queries where `actual_tool == expected_tool`.

**Example**

| Query | Expected | Actual | Correct? |
|-------|----------|--------|----------|
| `What is chain-of-thought prompting?` | `vector_db` | `vector_db` | ✅ |
| `Hello, how are you?` | `llm_direct` | `vector_db` | ❌ |

→ **accuracy = 1/2 = 0.5**.

### Retrieval Method Accuracy

**Definition:** Among queries that need retrieval, fraction where `actual_method == expected_method` (e.g. hybrid vs multi-query).

**Example:** expected `multi_query` for a compare question, router returns `hybrid_search` → **miss** for that query.

### Tool Call Success Rate

**Definition:** Fraction of tool invocations that completed without error (`success=True`).

**Example:** 3 routes chosen correctly, but greeting path returns no retrieval call / empty success flag → success rate can be **< 1.0** even when selection accuracy is perfect.

---

## Tool Use in Agents: What Actually Happens

```
User query → [Plan] → select tool → pass arguments → execute → observe result → (repeat) → final answer
```

| Decision | Example failure | What you'd log |
|----------|-----------------|----------------|
| **Which tool?** | Web search for a question answerable from your vector DB | `expected_tool` vs `actual_tool` |
| **Which variant?** | Dense search when the query needs multi-query expansion | `expected_method` vs `actual_method` |
| **Did it run?** | API timeout, bad credentials, malformed args | `success: true/false` |

> **Scope in this repo:** `ToolEvaluator` scores tool selection, method accuracy, and call success via `ToolTrace` objects.

---

## Demo: Semantic Router from Tutorial 1B

Our live demo uses the **semantic routing layer** from Tutorial 1B (`routing.py`) - a lightweight tool-selection step: *given this query, which tool (and retrieval method) should run?*

| Query type | Expected tool | Expected method |
|------------|---------------|-----------------|
| Factual question | `vector_db` | `hybrid_search` |
| Comparison / multi-facet | `vector_db` | `multi_query` |
| Greeting / chitchat | `llm_direct` | *(none)* |

> 🔧 **Fix playbook:** low tool selection → tune router; low method accuracy → refine sub-strategy rules; low success rate → check APIs / schemas first.

See **`tool_metrics.py`** for `ToolEvaluator`, `ToolTrace`, and `build_traces_from_routing()`.


In [ ]:
# Labeled routing cases: expected tool/method for sample queries
# (aligned with ROUTE_METADATA in routing.py)
routing_cases = [
    {
        "query": "What is chain-of-thought prompting?",
        "expected_tool": "vector_db",
        "expected_method": "hybrid_search",
    },
    {
        "query": "Compare RAG and fine-tuning approaches",
        "expected_tool": "vector_db",
        "expected_method": "multi_query",
    },
    {
        "query": "Hello, how are you?",
        "expected_tool": "llm_direct",
        "expected_method": None,
    },
]

tool_evaluator = ToolEvaluator()
tool_traces = ToolEvaluator.build_traces_from_routing(routing_cases, semantic_layer)
tool_result = tool_evaluator.evaluate(tool_traces)

print("=== Tool / Routing Metrics ===")
for name, score in tool_result.scores.items():
    print(f"  {name}: {score:.3f}")

print("\n=== Per-Query Routing Decisions ===")
for trace in tool_traces:
    match = "✅" if trace.actual_tool == trace.expected_tool else "❌"
    print(f"  {match} {trace.query[:50]}... → {trace.actual_tool} (expected: {trace.expected_tool})")

---

# Layer 4: Custom Metrics

## Why Extend Beyond Built-ins?

Off-the-shelf metrics cover general RAG quality, but production systems often need **domain-specific signals** - regulatory citations, token budgets, mandatory keywords, etc.

The helpers below are already used in Layers 1–2; this section shows how to call them directly.

### `keyword_overlap` (already defined in Layer 1)

Lexical overlap between retrieved chunks and gold evidence.  
**Example:** gold mentions `TPU` + `matrix`; chunks only say `GPU` → low score.

### `answer_length_ratio` (already defined in Layer 2)

`len(answer) / len(gold)`.  
**Example:** gold 100 chars, answer 500 chars → ratio `5.0`.

**Lightweight helpers** live in **`base.py`**:

```python
from retrieval_playground.src.evaluation.base import keyword_overlap, answer_length_ratio
```

For RAGAS-native custom metrics, subclass `SingleTurnMetric` - see the [RAGAS custom metrics guide](https://docs.ragas.io/en/stable/howtos/customizations/metrics/_write_your_own_metric/).


In [ ]:
from retrieval_playground.src.evaluation.base import keyword_overlap, answer_length_ratio

print(f"keyword_overlap (sample): {keyword_overlap(retrieved_contexts[0], reference_contexts[0]):.3f}")
print(f"answer_length_ratio (sample): {answer_length_ratio(answers[0], ground_truths[0]):.3f}")

### RAGAS custom metric template

```python
from dataclasses import dataclass, field
from ragas.metrics.base import SingleTurnMetric, MetricType

@dataclass
class MyRagasMetric(SingleTurnMetric):
    name: str = "my_ragas_metric"
    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {"user_input", "response"}
    })

    async def _single_turn_ascore(self, sample, callbacks) -> float:
        return 1.0  # your scoring logic
```

| Design choice | Recommendation |
|---------------|----------------|
| Metric name | Use `snake_case`, unique across your eval suite |
| Return value | Float in **0.0–1.0** range for consistency with RAGAS |
| Batch vs single | Use `base.py` helpers for batch heuristics; RAGAS calls `_single_turn_ascore()` per sample |


---

# ⚖️ Pipeline Comparison: Baseline vs Pre / Mid / Post

Each optimisation stage runs in its **own titled section** below with **self-contained code** (own imports + scoring). Run any arm alone after Setup + Load Test Data.

| Section | What it measures |
|---------|------------------|
| **Baseline RAG** | Dense retrieve (`k=3`) → generate |
| **Pre-Retrieval Optimisation** | `expand_query()` before dense retrieve |
| **Mid-Retrieval Optimisation** | Hybrid search (BM25 + dense + RRF) |
| **Post-Retrieval Optimisation** | Grade + refine on baseline chunks (compression off) |
| **Side-by-Side Summary** | Combines scores from arms you already ran |

> Mid/post can score **lower** than baseline when dense retrieval is already strong - they reshape ranking / filter context. Use them when Layer 1 shows keyword misses or noisy windows, not as always-on boosters.


## 1️⃣ Baseline RAG Evaluation

Dense retrieval only - the anchor scorecard everything else compares to.

Uses `rag.retrieve_context` on the default collection, then `document_assembly.generate_answer`, then RAGAS context + generation metrics.


In [ ]:
from retrieval_playground.src.post_retrieval import document_assembly
from retrieval_playground.src.evaluation import RAGEvaluator, GenerationEvaluator

K = 3
CONTEXT_METRICS = ["context_precision", "context_recall"]
GEN_METRICS = ["faithfulness", "answer_accuracy"]
BASELINE_METRIC_NAMES = CONTEXT_METRICS + GEN_METRICS + ["answer_length_ratio"]

baseline_queries = test_queries
baseline_ground_truths = [q["reference"] for q in baseline_queries]

baseline_results = []
for query_data in baseline_queries:
    query = query_data["user_input"]
    chunks = [doc for doc, _ in rag.retrieve_context(query, k=K)]
    baseline_results.append({
        "question": query,
        "answer": document_assembly.generate_answer(query, chunks),
        "context": [{"content": doc.page_content} for doc in chunks],
    })

baseline_ctx = RAGEvaluator(metrics=CONTEXT_METRICS).evaluate_rag_results(
    baseline_results, baseline_ground_truths
)
baseline_gen = GenerationEvaluator(ragas_metrics=GEN_METRICS).evaluate_rag_results(
    baseline_results, baseline_ground_truths
).scores
baseline_scores = {**baseline_ctx, **baseline_gen}

print("\n=== Baseline RAG Scores ===")
for metric in BASELINE_METRIC_NAMES:
    print(f"  {metric}: {baseline_scores[metric]:.3f}")


## 2️⃣ Pre-Retrieval Optimisation Evaluation

Runs `expand_query()` (Tutorial 1B) **before** dense retrieve → generate.


In [ ]:
# === Pre-Retrieval optimisation (self-contained) ===
from retrieval_playground.src.pre_retrieval.query_rephrasing import expand_query
from retrieval_playground.src.post_retrieval import document_assembly
from retrieval_playground.src.evaluation import RAGEvaluator, GenerationEvaluator

K = 3
CONTEXT_METRICS = ["context_precision", "context_recall"]
GEN_METRICS = ["faithfulness", "answer_accuracy"]
PRE_METRIC_NAMES = CONTEXT_METRICS + GEN_METRICS + ["answer_length_ratio"]

pre_queries = test_queries
pre_ground_truths = [q["reference"] for q in pre_queries]

pre_results = []
for query_data in pre_queries:
    query = query_data["user_input"]
    enhanced_query = expand_query(query, num_variants=1)[0]
    chunks = [doc for doc, _ in rag.retrieve_context(enhanced_query, k=K)]
    pre_results.append({
        "question": query,
        "answer": document_assembly.generate_answer(query, chunks),
        "context": [{"content": doc.page_content} for doc in chunks],
    })
    print(f"Original: {query}...")
    print(f"Expanded: {enhanced_query}\n")

pre_ctx = RAGEvaluator(metrics=CONTEXT_METRICS).evaluate_rag_results(
    pre_results, pre_ground_truths
)
pre_gen = GenerationEvaluator(ragas_metrics=GEN_METRICS).evaluate_rag_results(
    pre_results, pre_ground_truths
).scores
pre_scores = {**pre_ctx, **pre_gen}

print("=== Pre-Retrieval Scores ===")
for metric in PRE_METRIC_NAMES:
    print(f"  {metric}: {pre_scores[metric]:.3f}")


## 3️⃣ Mid-Retrieval Optimisation Evaluation

Uses `HybridRetriever` (BM25 + dense + RRF) instead of dense-only search — Tutorial 2.

Independent of baseline / pre cells. Requires the `hybrid` Qdrant collection.


In [ ]:
# === Mid-Retrieval optimisation (self-contained) ===
from retrieval_playground.src.mid_retrieval.hybrid_search import HybridRetriever
from retrieval_playground.src.post_retrieval import document_assembly
from retrieval_playground.src.evaluation import RAGEvaluator, GenerationEvaluator

K = 3
CONTEXT_METRICS = ["context_precision", "context_recall"]
GEN_METRICS = ["faithfulness", "answer_accuracy"]
MID_METRIC_NAMES = CONTEXT_METRICS + GEN_METRICS + ["answer_length_ratio"]

mid_queries = test_queries
mid_ground_truths = [q["reference"] for q in mid_queries]

hybrid = HybridRetriever(collection_name="hybrid")
mid_results = []
for query_data in mid_queries:
    query = query_data["user_input"]
    chunks = hybrid.search(query, k=K)
    mid_results.append({
        "question": query,
        "answer": document_assembly.generate_answer(query, chunks),
        "context": [{"content": doc.page_content} for doc in chunks],
    })
    top_rrf = chunks[0].metadata.get("rrf_score", 0) if chunks else 0
    print(f"{query[:70]}... | top rrf: {top_rrf:.4f}")

hybrid.close()

mid_ctx = RAGEvaluator(metrics=CONTEXT_METRICS).evaluate_rag_results(
    mid_results, mid_ground_truths
)
mid_gen = GenerationEvaluator(ragas_metrics=GEN_METRICS).evaluate_rag_results(
    mid_results, mid_ground_truths
).scores
mid_scores = {**mid_ctx, **mid_gen}

print("\n=== Mid-Retrieval Scores ===")
for metric in MID_METRIC_NAMES:
    print(f"  {metric}: {mid_scores[metric]:.3f}")


## 4️⃣ Post-Retrieval Optimisation Evaluation

Applies Tutorial 3 **grading + refinement** on dense-retrieved chunks before generation.

Compression is **off** so we measure filtering/refinement without empty-context failures. Independent of other arms.


In [ ]:
# === Post-Retrieval optimisation (self-contained) ===
from retrieval_playground.src.post_retrieval import ContextPreparer, document_assembly
from retrieval_playground.src.evaluation import RAGEvaluator, GenerationEvaluator

K = 3
CONTEXT_METRICS = ["context_precision", "context_recall"]
GEN_METRICS = ["faithfulness", "answer_accuracy"]
POST_METRIC_NAMES = CONTEXT_METRICS + GEN_METRICS + ["answer_length_ratio"]

post_queries = test_queries
post_ground_truths = [q["reference"] for q in post_queries]

preparer = ContextPreparer(run_compression=False)
post_results = []
for query_data in post_queries:
    query = query_data["user_input"]
    baseline_chunks = [doc for doc, _ in rag.retrieve_context(query, k=K)]
    prepared = preparer.prepare(query, baseline_chunks)
    post_results.append({
        "question": query,
        "answer": document_assembly.generate_answer(query, prepared.chunks),
        "context": [{"content": doc.page_content} for doc in prepared.chunks],
    })
    print(
        f"{query[:70]}... | chunks {len(baseline_chunks)} → {len(prepared.chunks)} "
        f"| tokens {prepared.token_before} → {prepared.token_after}"
    )

post_ctx = RAGEvaluator(metrics=CONTEXT_METRICS).evaluate_rag_results(
    post_results, post_ground_truths
)
post_gen = GenerationEvaluator(ragas_metrics=GEN_METRICS).evaluate_rag_results(
    post_results, post_ground_truths
).scores
post_scores = {**post_ctx, **post_gen}

print("\n=== Post-Retrieval Scores ===")
for metric in POST_METRIC_NAMES:
    print(f"  {metric}: {post_scores[metric]:.3f}")


## 📊 Side-by-Side Summary (optional)

In [ ]:
# === Combine arm scores that have already been run ===
METRIC_ORDER = [
    "context_precision",
    "context_recall",
    "faithfulness",
    "answer_accuracy",
    "answer_length_ratio",
]

arm_scores = {
    "Baseline": globals().get("baseline_scores"),
    "Pre-Retrieval": globals().get("pre_scores"),
    "Mid-Retrieval": globals().get("mid_scores"),
    "Post-Retrieval": globals().get("post_scores"),
}

available = {name: scores for name, scores in arm_scores.items() if scores}

comparison_df = None
if not available:
    print("No arm scores found. Run at least one of the Baseline / Pre / Mid / Post cells first.")
else:
    rows = []
    for metric in METRIC_ORDER:
        row = {"Metric": metric}
        for arm_name, scores in available.items():
            row[arm_name] = round(scores[metric], 3)
        rows.append(row)
    comparison_df = pd.DataFrame(rows)
    print("=== Pipeline comparison (arms run so far) ===")

comparison_df

---

## Hands-On: Evaluate Your Own Query

Everything above runs on a fixed test slice. This section lets you **point the full evaluation stack at any query** during a live demo.

**How to use during presentation:**

1. Uncomment the example call at the bottom of the cell
2. Swap in your own question and gold labels
3. Read the scores - which stage failed?

| Score pattern | Diagnosis |
|---------------|-----------|
| Low Hit@k, good faithfulness | Retriever missed evidence; generation made best of bad context |
| Good Hit@k, low faithfulness | Retrieved well but model hallucinated anyway |
| Good faithfulness, low answer accuracy | Grounded in context but still wrong vs gold reference |


In [ ]:
def evaluate_query(query: str, reference: str, reference_context: str):
    """End-to-end eval for a single custom query."""
    result = rag.query(query)
    contexts = [[ctx["content"] for ctx in result["context"]]]

    retrieval = RetrievalEvaluator(k=3).evaluate(
        [query], contexts, [reference_context]
    )
    generation = GenerationEvaluator().evaluate(
        [query], [result["answer"]], contexts, [reference]
    )

    print(f"Query: {query}\n")
    print("Retrieval:", {k: round(v, 3) for k, v in retrieval.scores.items()})
    print("Generation:", {k: round(v, 3) for k, v in generation.scores.items()})
    print(f"\nAnswer: {result['answer'][:300]}...")

# Example
evaluate_query(
    test_queries[1]["user_input"],
    test_queries[1]["reference"],
    test_queries[1]["reference_context"],
)

---

## 🎯 Conclusion: Evaluation Is a Diagnostic Stack


### What We Covered

| Layer | Key metrics | Module |
|-------|-------------|--------|
| Retrieval | Hit@k, MRR, context precision/recall | `retrieval_metrics.py` |
| Generation | Faithfulness, answer accuracy, length ratio | `generation_metrics.py` |
| Tools / Agents | Tool selection, method accuracy, call success | `tool_metrics.py` |
| Custom | Your domain-specific checks | `base.py` helpers + RAGAS `SingleTurnMetric` |
| Pipeline | Unified scorecard | `pipeline.py` |

### Key Takeaways

1. **Diagnose by stage** - a low aggregate score tells you nothing; a low *stage* score tells you where to invest engineering time.
2. **Use classical metrics for speed, RAGAS for nuance** - run cheap checks in CI, run LLM-judged metrics on release candidates.
3. **Compare pipelines stage by stage** - the baseline vs pre / mid / post table shows which layer actually helps (and which hurts when chained blindly).
4. **Compare chunking strategies in retrieval** - the same query can score very differently across recursive, contextual, parent-child, and docling indexes.
5. **Extend with custom metrics** - when off-the-shelf metrics miss your domain, use `base.py` helpers for fast heuristics or subclass RAGAS `SingleTurnMetric` for LLM-judged checks.

### Further Reading

- [RAGAS metrics reference](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- Custom metrics guide: `src/evaluation/EVALUATION.md`